# Installing required modules.


In [ ]:
!pip install torch torchvision onnx onnxruntime tvm onnxscript

# Cloning the LPRNet model repository


In [ ]:
!git clone https://github.com/sirius-ai/LPRNet_Pytorch.git

fatal: destination path 'LPRNet_Pytorch' already exists and is not an empty directory.


# Importing required libraries and setting up the working directory path.

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
sys.path.append(os.path.abspath('/content/LPRNet_Pytorch/model/'))
import torch.nn.utils.prune as prune
from torch.utils.data import DataLoader, Dataset
from imutils import paths
import numpy as np
import pandas as pd
import random
import cv2
import copy
from torch.quantization import quantize_dynamic
import torch.quantization as quant
import time
from onnxruntime.quantization import quantize_dynamic, QuantType

# Setting up the LPRNet model and builder in "eval" phase

In [ ]:
class small_basic_block(nn.Module):
    def __init__(self, ch_in, ch_out):
        super(small_basic_block, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(ch_in, ch_out // 4, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(ch_out // 4, ch_out // 4, kernel_size=(3, 1), padding=(1, 0)),
            nn.ReLU(),
            nn.Conv2d(ch_out // 4, ch_out // 4, kernel_size=(1, 3), padding=(0, 1)),
            nn.ReLU(),
            nn.Conv2d(ch_out // 4, ch_out, kernel_size=1),
        )
    def forward(self, x):
        return self.block(x)

class LPRNet(nn.Module):
    def __init__(self, lpr_max_len, phase, class_num, dropout_rate):
        super(LPRNet, self).__init__()
        self.phase = phase
        self.lpr_max_len = lpr_max_len
        self.class_num = class_num
        self.backbone = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, stride=1), # 0
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(),  # 2
            nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 1, 1)),
            small_basic_block(ch_in=64, ch_out=128),    # *** 4 ***
            nn.BatchNorm2d(num_features=128),
            nn.ReLU(),  # 6
            nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(2, 1, 2)),
            small_basic_block(ch_in=64, ch_out=256),   # 8
            nn.BatchNorm2d(num_features=256),
            nn.ReLU(),  # 10
            small_basic_block(ch_in=256, ch_out=256),   # *** 11 ***
            nn.BatchNorm2d(num_features=256),   # 12
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(4, 1, 2)),  # 14
            nn.Dropout(dropout_rate),
            nn.Conv2d(in_channels=64, out_channels=256, kernel_size=(1, 4), stride=1),  # 16
            nn.BatchNorm2d(num_features=256),
            nn.ReLU(),  # 18
            nn.Dropout(dropout_rate),
            nn.Conv2d(in_channels=256, out_channels=class_num, kernel_size=(13, 1), stride=1), # 20
            nn.BatchNorm2d(num_features=class_num),
            nn.ReLU(),  # *** 22 ***
        )
        self.container = nn.Sequential(
            nn.Conv2d(in_channels=448+self.class_num, out_channels=self.class_num, kernel_size=(1, 1), stride=(1, 1)),
            # nn.BatchNorm2d(num_features=self.class_num),
            # nn.ReLU(),
            # nn.Conv2d(in_channels=self.class_num, out_channels=self.lpr_max_len+1, kernel_size=3, stride=2),
            # nn.ReLU(),
        )

    def forward(self, x):
        keep_features = list()
        for i, layer in enumerate(self.backbone.children()):
            x = layer(x)
            if i in [2, 6, 13, 22]: # [2, 4, 8, 11, 22]
                keep_features.append(x)

        global_context = list()
        for i, f in enumerate(keep_features):
            if i in [0, 1]:
                f = nn.AvgPool2d(kernel_size=5, stride=5)(f)
            if i in [2]:
                f = nn.AvgPool2d(kernel_size=(4, 10), stride=(4, 2))(f)
            f_pow = torch.pow(f, 2)
            f_mean = torch.mean(f_pow)
            f = torch.div(f, f_mean)
            global_context.append(f)

        x = torch.cat(global_context, 1)
        x = self.container(x)
        logits = torch.mean(x, dim=2)

        return logits

def build_lprnet(lpr_max_len=8, phase=False, class_num=66, dropout_rate=0.5):

    Net = LPRNet(lpr_max_len, phase, class_num, dropout_rate)

    if phase == "train":
        return Net.train()
    else:
        return Net.eval()

lprnet = build_lprnet().eval()


# Setting up the LPRDataLoader and related functionalities.

In [ ]:
# Character Set for License Plates
CHARS = ['京', '沪', '津', '渝', '冀', '晋', '蒙', '辽', '吉', '黑',
         '苏', '浙', '皖', '闽', '赣', '鲁', '豫', '鄂', '湘', '粤',
         '桂', '琼', '川', '贵', '云', '藏', '陕', '甘', '青', '宁',
         '新', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
         'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'J', 'K',
         'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'U', 'V',
         'W', 'X', 'Y', 'Z', 'I', 'O', '-']

CHARS_DICT = {char: i for i, char in enumerate(CHARS)}

# Dataset Class
class LPRDataLoader(Dataset):
    def __init__(self, img_dir, imgSize, lpr_max_len, PreprocFun=None):
        self.img_dir = img_dir
        self.img_paths = []
        for dir_path in img_dir:
            self.img_paths += [el for el in paths.list_images(dir_path)]
        random.shuffle(self.img_paths)
        self.img_size = imgSize
        self.lpr_max_len = lpr_max_len
        self.PreprocFun = PreprocFun if PreprocFun else self.transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, index):
        filename = self.img_paths[index]
        image = cv2.imread(filename)
        image = cv2.resize(image, self.img_size)
        image = self.PreprocFun(image)

        basename = os.path.basename(filename)
        imgname = basename.split("-")[0].split("_")[0]

        imgname = ''.join([c for c in imgname if c in CHARS_DICT])

        label = [CHARS_DICT[c] for c in imgname]

        return torch.tensor(image, dtype=torch.float32), torch.tensor(label, dtype=torch.long), len(label)

    def transform(self, img):
        img = img.astype('float32')
        img -= 127.5
        img *= 0.0078125
        img = np.transpose(img, (2, 0, 1))  # Convert to (C, H, W)
        return img

def greedy_decode_evaluate(model, test_loader, device='cuda'):
    is_pytorch_model = isinstance(model, torch.nn.Module)
    if is_pytorch_model:
        model.eval().to(device)

    correct, total = 0, 0
    total_inference_time = 0

    with torch.no_grad():
        for data in test_loader:
            inputs, labels, lengths = data
            inputs = inputs.to(device)
            labels = labels.to(device)

            start_time = time.perf_counter()

            if not is_pytorch_model:
                inputs_numpy = inputs.cpu().numpy()
                model.set_input("data", inputs)  # Set input data
                model.run()  # Execute the model
                outputs = model.get_output(0)
                outputs = torch.from_numpy(outputs.numpy())
            else:
                outputs = model(inputs).cpu().numpy()  # PyTorch model inference


            end_time = time.perf_counter()

            inference_time = end_time - start_time
            total_inference_time += inference_time

            for i, preb in enumerate(outputs):
                preb_label = [np.argmax(preb[:, j]) for j in range(preb.shape[1])]

                no_repeat_blank_label = []
                prev_c = preb_label[0]
                if prev_c != len(CHARS) - 1:  # If not blank
                    no_repeat_blank_label.append(prev_c)
                for c in preb_label:
                    if c != prev_c and c != len(CHARS) - 1:
                        no_repeat_blank_label.append(c)
                    prev_c = c

                gt_label = labels[i][:lengths[i]].tolist()
                if no_repeat_blank_label == gt_label:
                    correct += 1
                total += 1

    accuracy = 100.0 * correct / total if total > 0 else 0.0
    avg_inference_time = (total_inference_time / len(test_loader)) * 1000

    print(f"Accuracy: {accuracy:.2f}% | Avg Inference Time: {avg_inference_time:.2f} ms")
    return accuracy, avg_inference_time

def load_pretrained_weights(model, pretrained_model_path):
    checkpoint = torch.load(pretrained_model_path)
    model_state_dict = model.state_dict()
    for key in checkpoint.keys():
        if key in model_state_dict and checkpoint[key].shape == model_state_dict[key].shape:
            model_state_dict[key] = checkpoint[key]
        else:
            print(f"Skipping layer {key} due to shape mismatch")
    model.load_state_dict(model_state_dict)
    print("Pre-trained weights loaded successfully!")

def get_test_loader(test_folder, batch_size=32):
    test_dataset = LPRDataLoader(
        img_dir=test_folder.split(','),
        imgSize=(94, 24),
        lpr_max_len=8
    )
    return DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# Setting up a function to port the trained model to ONXX and return File Size

In [ ]:
def calculate_onnx_model_size(model, dummy_input, file_path):
    torch.onnx.export(
        model,
        dummy_input,
        file_path,
        weight_type=QuantType.QInt8
    )

    model_size_bytes = os.path.getsize(file_path)  # File size in bytes
    model_size_mb = model_size_bytes / (1024 * 1024)  # Convert to megabytes

    print(f"Model saved to {file_path}. Size: {model_size_mb:.2f} MB")
    return model_size_mb

In [ ]:
def get_model_size(model):
    param_size = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.numel() * b.element_size() for b in model.buffers())
    total_size = param_size + buffer_size
    return total_size / (1024 ** 2)  # Convert to MB

In [ ]:
onnx_file_path = "exported_model.onnx"
onnx_model_path = '/content/LPRNet_Pytorch/weights/Final_LPRNet_model.pth'
dummy_input = torch.randn(1, 3, 24, 94, device="cuda")
lprnet.to(dummy_input.device)
model_size = calculate_onnx_model_size(lprnet, dummy_input, onnx_file_path)
print(f"Model size in memory: {get_model_size(lprnet):.2f} MB")

Model saved to exported_model.onnx. Size: 1.68 MB
Model size in memory: 1.68 MB


In [ ]:
if __name__ == "__main__":
    lprnet = build_lprnet(lpr_max_len=8, phase="test", class_num=66, dropout_rate=0.5).to('cuda')
    weights_path = '/content/LPRNet_Pytorch/weights/Final_LPRNet_model.pth'  # Update with the correct path
    test_folder = "/content/LPRNet_Pytorch/data/test"
    test_loader = get_test_loader(test_folder, batch_size=100)  # Load the test dataset
    lprnet = build_lprnet(class_num=68, dropout_rate=0.5)  # Replace with your model initialization
    lprnet.load_state_dict(torch.load(weights_path, weights_only=True))
    lprnet.to('cuda')
    print("Model loaded successfully!")
    test_accuracy,inference_time = greedy_decode_evaluate(lprnet, test_loader, device='cuda')
    print(f"Test Accuracy: {test_accuracy:.2f}%")

Model loaded successfully!
Accuracy: 89.90% | Avg Inference Time: 36.66 ms
Test Accuracy: 89.90%


# Setting up Layer fusion


In [ ]:
import torch
import torch.nn as nn

def lprnet_fusion(model):
    backbone = model.backbone

    supported_fusions = [
        ['0', '1', '2'],    # Conv-BN-ReLU
        ['16', '17', '18'], # Conv-BN-ReLU
        ['20', '21', '22']  # Conv-BN-ReLU
    ]

    for layers in supported_fusions:
        try:
            torch.quantization.fuse_modules(backbone, layers, inplace=True)
        except AssertionError as e:
            print(f"Skipping unsupported fusion: {layers} -> {e}")

    small_blocks_to_fuse = [4, 8, 11]
    for idx in small_blocks_to_fuse:
        block = backbone[idx].block
        layers_to_fuse = [
            ['0', '1'],  # Conv2d -> ReLU
            ['2', '3'],  # Conv2d -> ReLU
            ['4', '5']   # Conv2d -> ReLU
        ]
        try:
            torch.quantization.fuse_modules(block, layers_to_fuse, inplace=True)
        except AssertionError as e:
            print(f"Skipping unsupported fusion in block {idx}: {e}")

    return model

In [ ]:
import torch

def get_model_size(model):
    param_size = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.numel() * b.element_size() for b in model.buffers())
    total_size = (param_size + buffer_size) / (1024 ** 2)  # Convert bytes to MB
    return total_size

# Get the size of the fused LPRNet model
model_size = get_model_size(lprnet_fusion(lprnet))
print(f"Fused model size in memory: {model_size:.2f} MB")

Fused model size in memory: 1.71 MB


In [ ]:
fused_lprnet_model = lprnet_fusion(lprnet).to('cuda')
calculate_onnx_model_size(fused_lprnet_model, dummy_input, onnx_file_path)

Skipping unsupported fusion: ['0', '1', '2'] -> did not find fuser method for: (<class 'torch.ao.nn.intrinsic.modules.fused.ConvReLU2d'>, <class 'torch.nn.modules.linear.Identity'>, <class 'torch.nn.modules.linear.Identity'>) 
Skipping unsupported fusion: ['16', '17', '18'] -> did not find fuser method for: (<class 'torch.ao.nn.intrinsic.modules.fused.ConvReLU2d'>, <class 'torch.nn.modules.linear.Identity'>, <class 'torch.nn.modules.linear.Identity'>) 
Skipping unsupported fusion: ['20', '21', '22'] -> did not find fuser method for: (<class 'torch.ao.nn.intrinsic.modules.fused.ConvReLU2d'>, <class 'torch.nn.modules.linear.Identity'>, <class 'torch.nn.modules.linear.Identity'>) 
Skipping unsupported fusion in block 4: did not find fuser method for: (<class 'torch.ao.nn.intrinsic.modules.fused.ConvReLU2d'>, <class 'torch.nn.modules.linear.Identity'>) 
Skipping unsupported fusion in block 8: did not find fuser method for: (<class 'torch.ao.nn.intrinsic.modules.fused.ConvReLU2d'>, <class '

1.7072038650512695

In [ ]:
print(greedy_decode_evaluate(fused_lprnet_model, test_loader, device='cuda'))

Accuracy: 89.90% | Avg Inference Time: 28.22 ms
(89.9, 28.221796299953894)


# Setting up the Quantizations


##Dynamic Quantization

In [ ]:
def apply_dynamic_quantization(model):
    return quant.quantize_dynamic(model, {nn.Linear, nn.Conv2d}, dtype=torch.qint8)

In [ ]:
import time
import torch
import torch.nn as nn
import torch.quantization as quant

# Apply Dynamic Quantization
def apply_dynamic_quantization(model):
    quantized_model = quant.quantize_dynamic(
        model, {nn.Linear, nn.Conv2d}, dtype=torch.qint8
    )
    print("Dynamic Quantization applied!")
    return quantized_model

# Example Usage
if __name__ == '__main__':
    # Load test dataset with increased batch size
    test_loader = get_test_loader("/content/LPRNet_Pytorch/data/test", batch_size=32)

    # Evaluate Original Model
    lprnet.eval().to("cpu")
    original_time_start = time.time()
    original_accuracy,inference_time = greedy_decode_evaluate(lprnet, test_loader, device="cpu")
    original_time = (time.time() - original_time_start)

    # Apply Dynamic Quantization and Evaluate Quantized Model
    quantized_model = apply_dynamic_quantization(lprnet)
    quantized_time_start = time.time()
    quantized_model.eval().to("cpu")
    quantized_accuracy,inference_time = greedy_decode_evaluate(quantized_model, test_loader, device="cpu")
    quantized_time = (time.time() - quantized_time_start)
    # Print Evaluation Results
    print("\n=== Model Evaluation Results ===")
    print(f"Original Model -> Accuracy: {original_accuracy:.2f}% | Avg Inference Time: {original_time:.2f} ms")
    print(f"Quantized Model -> Accuracy: {quantized_accuracy:.2f}% | Avg Inference Time: {quantized_time:.2f} ms")


Accuracy: 90.00% | Avg Inference Time: 2325.89 ms
Dynamic Quantization applied!
Accuracy: 90.00% | Avg Inference Time: 2129.11 ms

=== Model Evaluation Results ===
Original Model -> Accuracy: 90.00% | Avg Inference Time: 74.97 ms
Quantized Model -> Accuracy: 90.00% | Avg Inference Time: 68.53 ms


##Post Training Static Quantization

In [ ]:
def test_model(model, test_img_dirs, img_size=(94, 24), lpr_max_len=8, batch_size=32, device='cpu'):
    """
    Evaluates the model on the test dataset using greedy decoding.
    """
    # Expand and validate test image directory
    test_img_dirs = os.path.expanduser(test_img_dirs)
    if not os.path.exists(test_img_dirs):
        raise ValueError(f"Test folder {test_img_dirs} does not exist.")

    # Load the test dataset using LPRDataLoader
    test_dataset = LPRDataLoader(
        img_dir=test_img_dirs.split(','),
        imgSize=img_size,
        lpr_max_len=lpr_max_len
    )

    # Initialize DataLoader
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    if len(test_loader) == 0:
        raise ValueError("Test dataset is empty. Please check the dataset path.")

    # Evaluate the model using greedy decoding
    accuracy, avg_inference_time = greedy_decode_evaluate(model, test_loader, device=device)
    print(f"Test Accuracy: {accuracy:.2f}% | Avg Inference Time: {avg_inference_time:.2f} ms")

    return accuracy, avg_inference_time


In [ ]:
if __name__ == "__main__":
    import torch.backends.quantized

    torch.backends.quantized.engine = 'fbgemm'

    model = LPRNet(lpr_max_len=8, phase="test", class_num=68, dropout_rate=0.5).to('cpu')
    weights_path = "/content/LPRNet_Pytorch/weights/Final_LPRNet_model.pth"       ## We should add the trained weight file path

    load_pretrained_weights(model, weights_path)
    test_img_dirs = "/content/LPRNet_Pytorch/data/test"
    test_model(model, test_img_dirs, img_size=(94, 24), lpr_max_len=8, batch_size=100, device='cpu')


Pre-trained weights loaded successfully!


<ipython-input-60-ccccf01daa4f>:104: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(pretrained_model_path)


Accuracy: 89.80% | Avg Inference Time: 12178.60 ms
Test Accuracy: 89.80% | Avg Inference Time: 12178.60 ms


# Pruning Functions

In [ ]:
# Pruning Function - Unstructured Sparse Pruning Based on Percentage
def apply_unstructured_pruning(model, pruning_percentage, device='cuda'):
    model_copy = copy.deepcopy(model).to(device)  # Create a copy of the model
    for name, module in model_copy.named_modules():
        if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
            module = prune.l1_unstructured(module, name="weight", amount=pruning_percentage)
            prune.remove(module, 'weight')
    print("Unstructured Percentage-Based Pruning complete!")
    return model_copy

def apply_kernel_pruning(model, pruning_percentage, device='cuda'):
    model_copy = copy.deepcopy(model).to(device)
    for name, module in model_copy.named_modules():
        if isinstance(module, nn.Conv2d):
            # Compute L2 norm for each kernel
            weight = module.weight.detach().cpu().numpy()
            kernel_l2_norm = torch.norm(torch.tensor(weight), p=2, dim=(2, 3))

            # Flatten to prune a fraction of kernels across all filters
            num_kernels = kernel_l2_norm.numel()
            num_pruned = int(pruning_percentage * num_kernels)
            if num_pruned == 0:
                continue
            threshold = torch.topk(kernel_l2_norm.flatten(), num_pruned, largest=False).values[-1]

            # Create a mask for kernels to keep
            mask = (kernel_l2_norm > threshold).float().to(device)

            # Apply the mask to prune kernels
            with torch.no_grad():
                module.weight *= mask.view(module.weight.size(0), module.weight.size(1), 1, 1)
    return model_copy

def apply_filter_pruning(model, pruning_percentage, device='cuda'):
    model_copy = copy.deepcopy(model).to(device)
    for name, module in model_copy.named_modules():
        if isinstance(module, nn.Conv2d):
            weight = module.weight.detach().cpu().numpy()
            l2_norm = torch.norm(torch.tensor(weight), p=2, dim=(1, 2, 3))

            # Determine the threshold for pruning
            num_filters = len(l2_norm)
            num_pruned = int(pruning_percentage * num_filters)
            if num_pruned == 0:
                continue
            threshold = torch.topk(l2_norm, num_pruned, largest=False).values[-1]

            # Create a mask for the filters to keep
            mask = (l2_norm > threshold).float().to(device)

            # Apply the mask to prune weights and biases
            with torch.no_grad():
                module.weight *= mask.view(-1, 1, 1, 1)
                if module.bias is not None:
                    module.bias *= mask
    print("Filter Pruning complete!")
    model_size = calculate_onnx_model_size(model_copy, dummy_input, onnx_file_path)
    print(f"ONNX Model Size After {name}: {model_size:.2f} MB")
    return model_copy

In [ ]:
def calculate_sparsity(model):
    nonzero_params = sum(torch.count_nonzero(p).item() for p in model.parameters())
    total_params = sum(p.numel() for p in model.parameters())
    sparsity = 100 * (1 - nonzero_params / total_params)
    return round(sparsity, 2)

def count_nonzero_params(model):
    nonzero = 0
    total = 0
    for param in model.parameters():
        nonzero += torch.count_nonzero(param).item()
        total += param.numel()
    print(f"Non-zero parameters: {nonzero}, Total parameters: {total}, Sparsity: {100 * (1 - nonzero/total):.2f}%")
    return nonzero, total, 100 * (1 - nonzero/total)

In [ ]:
if __name__ == '__main__':
    import time
    import torch
    import pandas as pd

    test_folder = "/content/LPRNet_Pytorch/data/test"  # Path to test dataset
    weights_path = '/content/LPRNet_Pytorch/weights/Final_LPRNet_model.pth'  # Pretrained weights path
    test_loader = get_test_loader(test_folder, batch_size=100)  # Load test data
    pruning_results = []

    # Load and Evaluate the Original Model
    lprnet.load_state_dict(torch.load(weights_path), strict=False)
    lprnet.eval().to('cuda')
    original_nonzero, original_total, original_sparsity = count_nonzero_params(lprnet)
    model_total = original_total

    # Measure Inference Time and Accuracy
    og_start_time = time.time()
    original_accuracy,inference_time = greedy_decode_evaluate(lprnet, test_loader, device='cuda')
    og_end_time = time.time()
    og_inference_time = (og_end_time - og_start_time)   # In milliseconds

    # Record Baseline Results
    pruning_results.append({
        "Model": "Baseline (No Pruning)",
        "Pruning (%)": "0%",
        "Accuracy (%)": round(original_accuracy, 2),
        "Total Params": model_total,
        "Non-Zero Params": original_nonzero,
        "Sparsity (%)": round(original_sparsity, 2),
        "Inference Time (ms)": round(og_inference_time, 2)
    })

    # Define Pruning Percentages to Evaluate
    pruning_percentages = [0.1, 0.2, 0.3, 0.45]

    # Apply Different Pruning Techniques for Multiple Percentages
    for percentage in pruning_percentages:
        print(f"\n=== Evaluating Pruning Percentage: {percentage*100:.1f}% ===\n")

        pruned_models = [
            ("Unstructured Pruning", apply_unstructured_pruning(lprnet, pruning_percentage=percentage, device='cuda')),
            ("Filter Pruning", apply_filter_pruning(lprnet, pruning_percentage=percentage, device='cuda')),
            ("Kernel Pruning", apply_kernel_pruning(lprnet, pruning_percentage=percentage, device='cuda'))
        ]

        # Evaluate Each Pruned Model
        for name, model in pruned_models:
            model.eval().to("cuda")

            # Measure Inference Time
            start_time = time.time()
            test_accuracy = greedy_decode_evaluate(model, test_loader, device='cuda')
            end_time = time.time()
            inference_time = (end_time - start_time)   # Convert to milliseconds

            # Evaluate and Save Metrics
            model_total = sum(p.numel() for p in model.parameters())
            model_nonzero, model_total, model_sparsity = count_nonzero_params(model)

            pruning_results.append({
                "Model": name,
                "Pruning (%)": f"{percentage*100:.1f}%",
                "Accuracy (%)": test_accuracy,
                "Total Params": model_total,
                "Non-Zero Params": model_nonzero,
                "Sparsity (%)": round(model_sparsity, 2),
                "Inference Time (ms)": round(inference_time, 2)
            })

    # Generate Results Table
    print("\n=== Pruning Results Table ===\n")
    results_df = pd.DataFrame(pruning_results)
    print(results_df.to_markdown(index=False))
    results_df.to_csv("pruning_results.csv", index=False)


Non-zero parameters: 323251, Total parameters: 446200, Sparsity: 27.55%


<ipython-input-75-d51cbd385547>:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  lprnet.load_state_dict(torch.load(weights_path), strict=False)


Accuracy: 89.90% | Avg Inference Time: 32.25 ms

=== Evaluating Pruning Percentage: 10.0% ===

Unstructured Percentage-Based Pruning complete!
Filter Pruning complete!
Model saved to exported_model.onnx. Size: 1.71 MB
ONNX Model Size After container.0: 1.71 MB
Accuracy: 90.10% | Avg Inference Time: 26.45 ms
Non-zero parameters: 308273, Total parameters: 446200, Sparsity: 30.91%
Accuracy: 16.60% | Avg Inference Time: 26.01 ms
Non-zero parameters: 282309, Total parameters: 446200, Sparsity: 36.73%
Accuracy: 89.10% | Avg Inference Time: 27.27 ms
Non-zero parameters: 308287, Total parameters: 446200, Sparsity: 30.91%

=== Evaluating Pruning Percentage: 20.0% ===

Unstructured Percentage-Based Pruning complete!
Filter Pruning complete!
Model saved to exported_model.onnx. Size: 1.71 MB
ONNX Model Size After container.0: 1.71 MB
Accuracy: 90.10% | Avg Inference Time: 26.05 ms
Non-zero parameters: 293294, Total parameters: 446200, Sparsity: 34.27%
Accuracy: 0.70% | Avg Inference Time: 26.49 ms

In [ ]:
final_pruned_model = apply_unstructured_pruning(lprnet, pruning_percentage=0.45, device='cuda')

Unstructured Percentage-Based Pruning complete!


# MLC Optimizations

#### Initializing for MLC Optimizations

In [ ]:
!!python3 -m  pip install mlc-ai-nightly -f https://mlc.ai/wheels
!pip install tvm

#### Importing Required Libraries

In [ ]:
import onnx
import tvm
from tvm import relay
from tvm.contrib import graph_executor
import datetime
from tvm import autotvm
from tvm.autotvm.tuner import XGBTuner


In [ ]:
import onnx
import tvm
from tvm import relay
from tvm.contrib import graph_executor

onnx_path = "lprnet.onnx"
input_shape = (1, 3, 24, 94)



device = torch.device("cpu")
x = torch.randn(input_shape, device=device) # Create input tensor on the appropriate device

fused_lprnet_model.to("cpu")

torch.onnx.export(
    fused_lprnet_model,
    x,
    onnx_path
)

onnx_model = onnx.load(onnx_path)
input_shape = (1,3,24,94)

input_name = onnx_model.graph.input[0].name

model,params = relay.frontend.from_onnx(onnx_model,shape={input_name: input_shape})

# Auto TVM Optimisation

target =  tvm.target.Target("llvm")
with tvm.transform.PassContext(opt_level=3):
    executor = relay.build(model, target, params=params)

# 4. Create runtime module
lib = executor




target = "llvm"

jit_model = torch.jit.trace(fused_lprnet_model,x).eval()

shape_info = [("data",(1,3,24,94))]

relay_model,params = relay.frontend.from_pytorch(jit_model,shape_info)

with tvm.transform.PassContext(opt_level=0):
    lib = relay.build(relay_model,target,params=params)

target = "llvm"
dev = tvm.device(str(target), 0)
graph_module = graph_executor.GraphModule(lib["default"](dev))

#### Relay Custom Optimizations

In [ ]:
def apply_custom_optimizations(mod):


    with tvm.target.Target("llvm"):

        # Operator Fusion Optimizations

        #Optimization 1 : Merge dense layers
        mod = tvm.relay.transform.CombineParallelDense()(mod)

        #Optimization 2 : Merge parallel conv2d
        mod = tvm.relay.transform.CombineParallelConv2D()(mod)


        # Loop level Optimizations

        #Optimization 3 : Loop Vectorization
        mod = tvm.tir.transform.VectorizeLoop()(mod)

        #Optimization 4 : Loop tiling (Partition Loops)
        mod = tvm.tir.transform.LoopPartition()(mod)

        # Inference Simplification Optimizations

        #Optimization 5 : Inference Simplification
        mod = tvm.relay.transform.SimplifyInference()(mod)

        #Optimization 6: Inline functions
        mod = tvm.relay.transform.Inline()(mod)


        # Expression Simplification Optimization

        #Optimization 7 : Remove redundant sub expressions
        mod = relay.transform.EliminateCommonSubexpr()(mod)

        #Optimization 8 : Fold constants
        # mod = tvm.relay.transform.FoldConstant()(mod)


    return mod


opt_mod = apply_custom_optimizations(relay_model)


target = tvm.target.Target("llvm") # Updated API call
with tvm.transform.PassContext(opt_level=0):
    opt_executor = relay.build(opt_mod, target, params=params)


device = tvm.device(str(target), 0)
custom_opt_model = graph_executor.GraphModule(opt_executor["default"](dev))


test_loader = get_test_loader("/content/LPRNet_Pytorch/data/test", batch_size=1)
original_accuracy, original_time = greedy_decode_evaluate(graph_module, test_loader,device="cpu")

opt_accuracy, opt_time = greedy_decode_evaluate(custom_opt_model, test_loader,device="cpu")

print("\n=== Model Evaluation Results ===")
print(f"Original Model -> Accuracy: {original_accuracy:.2f}% | Avg Inference Time: {original_time:.2f} ms")
print(f"Optimized Model -> Accuracy: {opt_accuracy:.2f}% | Avg Inference Time: {opt_time:.2f} ms")




Accuracy: 89.40% | Avg Inference Time: 159.40 ms
Accuracy: 89.40% | Avg Inference Time: 155.79 ms

=== Model Evaluation Results ===
Original Model -> Accuracy: 89.40% | Avg Inference Time: 159.40 ms
Optimized Model -> Accuracy: 89.40% | Avg Inference Time: 155.79 ms


### Auto TVM

In [ ]:
from tvm import autotvm

# Create tuning tasks for the target and Relay model
tasks = autotvm.task.extract_from_program(relay_model["main"], target=target, params=params)


from tvm.autotvm.tuner import XGBTuner

log_file = "tuning.log"

tuning_options = {
    "tuner": "xgb",
    "trials": 20,
    "early_stopping": 100,
    "measure_option": autotvm.measure_option(
        builder=autotvm.LocalBuilder(build_func="default"),
        runner=autotvm.LocalRunner(number = 10,repeat=1, min_repeat_ms=0, timeout=10,enable_cpu_cache_flush=True),
    ),
}


for i, task in enumerate(tasks):
    print(f"Tuning task {i + 1}/{len(tasks)}: {task.name}")
    tuner = XGBTuner(task,loss_type="reg")
    tuner.tune(
        n_trial=tuning_options["trials"],
        early_stopping=tuning_options["early_stopping"],
        measure_option=tuning_options["measure_option"],
        callbacks=[autotvm.callback.log_to_file(log_file),autotvm.callback.progress_bar(tuning_options["trials"])],
    )



Tuning task 1/13: conv2d_NCHWc.x86
 Current/Best:   14.18/  21.43 GFLOPS | Progress: (20/20) | 19.31 s Done.
Tuning task 2/13: conv2d_NCHWc.x86
 Current/Best:    5.74/  19.93 GFLOPS | Progress: (20/20) | 14.28 s Done.
Tuning task 3/13: conv2d_NCHWc.x86
 Current/Best:   16.55/  17.54 GFLOPS | Progress: (20/20) | 13.16 s Done.
Tuning task 4/13: conv2d_NCHWc.x86
 Current/Best:   14.32/  21.84 GFLOPS | Progress: (20/20) | 13.00 s Done.
Tuning task 5/13: conv2d_NCHWc.x86
 Current/Best:    5.80/  17.68 GFLOPS | Progress: (20/20) | 17.04 s Done.
Tuning task 6/13: conv2d_NCHWc.x86
 Current/Best:   13.05/  19.11 GFLOPS | Progress: (20/20) | 16.18 s Done.
Tuning task 7/13: conv2d_NCHWc.x86
 Current/Best:   13.30/  21.42 GFLOPS | Progress: (20/20) | 13.24 s Done.
Tuning task 8/13: conv2d_NCHWc.x86
 Current/Best:   19.33/  20.52 GFLOPS | Progress: (20/20) | 12.91 s Done.
Tuning task 9/13: conv2d_NCHWc.x86
 Current/Best:    2.60/  17.87 GFLOPS | Progress: (20/20) | 25.50 s Done.
Tuning task 10/13: 

In [ ]:
with autotvm.apply_history_best(log_file):
    with tvm.transform.PassContext(opt_level=0):
        auto_tuned_model = relay.build(relay_model, target, params=params)


device = tvm.device(str(target), 0)
optimized_model = graph_executor.GraphModule(auto_tuned_model["default"](dev))


test_loader = get_test_loader("/content/LPRNet_Pytorch/data/test", batch_size=1)
original_accuracy, original_time = greedy_decode_evaluate(graph_module, test_loader,device="cpu")

autotuned_accuracy, autotuned_time = greedy_decode_evaluate(optimized_model, test_loader,device="cpu")

print("\n=== Model Evaluation Results ===")
print(f"Original Model -> Accuracy: {original_accuracy:.2f}% | Avg Inference Time: {original_time:.2f} ms")
print(f"Autotuned Model -> Accuracy: {autotuned_accuracy:.2f}% | Avg Inference Time: {autotuned_time:.2f} ms")

Accuracy: 89.40% | Avg Inference Time: 178.28 ms
Accuracy: 89.40% | Avg Inference Time: 152.25 ms

=== Model Evaluation Results ===
Original Model -> Accuracy: 89.40% | Avg Inference Time: 178.28 ms
Autotuned Model -> Accuracy: 89.40% | Avg Inference Time: 152.25 ms


### Trials

In [ ]:
import onnx
import tvm
from tvm import relay
from tvm.contrib import graph_executor

onnx_path = "lprnet.onnx"
input_shape = (1, 3, 24, 94)



device = torch.device("cpu")
x = torch.randn(input_shape, device=device) # Create input tensor on the appropriate device


final_pruned_model.to("cpu")

torch.onnx.export(
    final_pruned_model,
    x,
    onnx_path
)

onnx_model = onnx.load(onnx_path)
input_shape = (1,3,24,94)

input_name = onnx_model.graph.input[0].name

model,params = relay.frontend.from_onnx(onnx_model,shape={input_name: input_shape})

# Auto TVM Optimisation

target =  tvm.target.Target("llvm")
with tvm.transform.PassContext(opt_level=3):
    executor = relay.build(model, target, params=params)

# 4. Create runtime module
lib = executor




target = "llvm"

jit_model = torch.jit.trace(final_pruned_model,x).eval()

shape_info = [("data",(1,3,24,94))]

relay_model,params = relay.frontend.from_pytorch(jit_model,shape_info)

with tvm.transform.PassContext(opt_level=0):
    lib = relay.build(relay_model,target,params=params)

target = "llvm"
dev = tvm.device(str(target), 0)
graph_module = graph_executor.GraphModule(lib["default"](dev))

In [ ]:
from tvm import autotvm

# Create tuning tasks for the target and Relay model
tasks = autotvm.task.extract_from_program(relay_model["main"], target=target, params=params)


from tvm.autotvm.tuner import XGBTuner

log_file = "tuning.log"

tuning_options = {
    "tuner": "xgb",
    "trials": 20,
    "early_stopping": 100,
    "measure_option": autotvm.measure_option(
        builder=autotvm.LocalBuilder(build_func="default"),
        runner=autotvm.LocalRunner(number = 10,repeat=1, min_repeat_ms=0, timeout=10,enable_cpu_cache_flush=True),
    ),
}


for i, task in enumerate(tasks):
    print(f"Tuning task {i + 1}/{len(tasks)}: {task.name}")
    tuner = XGBTuner(task,loss_type="reg")
    tuner.tune(
        n_trial=tuning_options["trials"],
        early_stopping=tuning_options["early_stopping"],
        measure_option=tuning_options["measure_option"],
        callbacks=[autotvm.callback.log_to_file(log_file),autotvm.callback.progress_bar(tuning_options["trials"])],
    )



Tuning task 1/13: conv2d_NCHWc.x86
 Current/Best:   12.70/  18.15 GFLOPS | Progress: (20/20) | 20.54 s Done.
Tuning task 2/13: conv2d_NCHWc.x86
 Current/Best:   13.04/  20.61 GFLOPS | Progress: (20/20) | 15.20 s Done.
Tuning task 3/13: conv2d_NCHWc.x86
 Current/Best:   14.70/  19.60 GFLOPS | Progress: (20/20) | 13.15 s Done.
Tuning task 4/13: conv2d_NCHWc.x86
 Current/Best:   16.40/  19.77 GFLOPS | Progress: (20/20) | 13.13 s Done.
Tuning task 5/13: conv2d_NCHWc.x86
 Current/Best:   15.98/  17.52 GFLOPS | Progress: (20/20) | 12.88 s Done.
Tuning task 6/13: conv2d_NCHWc.x86
 Current/Best:    3.79/  15.15 GFLOPS | Progress: (20/20) | 14.76 s Done.
Tuning task 7/13: conv2d_NCHWc.x86
 Current/Best:   19.09/  21.91 GFLOPS | Progress: (20/20) | 12.67 s Done.
Tuning task 8/13: conv2d_NCHWc.x86
 Current/Best:   19.91/  19.91 GFLOPS | Progress: (20/20) | 12.79 s Done.
Tuning task 9/13: conv2d_NCHWc.x86
 Current/Best:   10.72/  19.05 GFLOPS | Progress: (20/20) | 29.00 sTuning task 10/13: conv2d_